# 🧠 Notebook 03 – QLoRA Fine-Tuning

**SpaceSLM-mini** | Training Step

> ⚠️ **This notebook is designed for Google Colab with a GPU runtime.**  
> Go to `Runtime → Change runtime type → T4 GPU` before running.

---

### 🎯 What you will do
1. Set up the Colab GPU environment
2. Load `Qwen2.5-0.5B-Instruct` in 4-bit NF4 quantization
3. Attach LoRA adapters to attention and MLP layers
4. Fine-tune on the space-domain instruction dataset
5. Save the trained adapter
6. Merge the adapter and export the model

### 📚 Key concepts
- **QLoRA**: Quantized Low-Rank Adaptation – efficient fine-tuning on consumer GPUs
- **NF4**: Normal Float 4-bit – the best quantization type for language models
- **LoRA rank (r)**: Controls adapter size. Higher r = more capacity but more VRAM
- **LoRA alpha**: Scaling factor. Rule of thumb: alpha = 2 × r
- **SFTTrainer**: Supervised Fine-Tuning Trainer from TRL – handles the training loop


In [ ]:
# ── Step 0: Check GPU ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU: {gpu_name}  ({vram_gb:.1f} GB VRAM)')
else:
    print('❌ No GPU detected!')
    print('Go to: Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
# Run this cell once at the start of a Colab session
import subprocess, sys
packages = [
    'transformers>=4.40.0',
    'peft>=0.10.0',
    'trl>=0.8.6',
    'bitsandbytes>=0.43.0',
    'accelerate>=0.29.0',
    'datasets>=2.18.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + packages)
print('✅ Packages installed')

In [ ]:
# ── Step 2: Mount Google Drive (if using Drive for data/output) ───────────────
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/SpaceSLM'

# Or clone the repo directly:
# !git clone https://github.com/hdq4ch/SpaceSLM.git
# %cd SpaceSLM

import os
# os.chdir('/content/SpaceSLM')  # adjust to your path
print('Working directory:', os.getcwd())

In [ ]:
# ── Step 3: Imports ───────────────────────────────────────────────────────────
import json
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments,
)
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print('✅ All imports OK')

In [ ]:
# ── Step 4: Configuration ─────────────────────────────────────────────────────
MODEL_ID       = 'Qwen/Qwen2.5-0.5B-Instruct'
ADAPTER_OUTPUT = 'models/adapter'
TRAIN_FILE     = 'data/processed/train.jsonl'
VAL_FILE       = 'data/processed/val.jsonl'
MAX_SEQ_LEN    = 512

# LoRA hyperparameters
LORA_R         = 8      # rank – controls adapter size
LORA_ALPHA     = 16     # scaling factor
LORA_DROPOUT   = 0.05

# Training hyperparameters
EPOCHS         = 3
BATCH_SIZE     = 4
GRAD_ACCUM     = 4      # effective batch = BATCH_SIZE × GRAD_ACCUM = 16
LEARNING_RATE  = 2e-4

print(f'Base model: {MODEL_ID}')
print(f'LoRA: r={LORA_R}, alpha={LORA_ALPHA}')
print(f'Training: {EPOCHS} epochs, lr={LEARNING_RATE}')

In [ ]:
# ── Step 5: Load Dataset ──────────────────────────────────────────────────────
def load_jsonl(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_jsonl(TRAIN_FILE)
val_records   = load_jsonl(VAL_FILE)

train_dataset = Dataset.from_list(train_records)
val_dataset   = Dataset.from_list(val_records)

print(f'Train: {len(train_dataset)} examples')
print(f'Val  : {len(val_dataset)} examples')
print('\nSample text (first 500 chars):')
print(train_dataset[0]['text'][:500])

In [ ]:
# ── Step 6: Load Tokenizer ────────────────────────────────────────────────────
print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Vocabulary size: {tokenizer.vocab_size:,}')
print(f'Pad token: {tokenizer.pad_token!r}')

In [ ]:
# ── Step 7: 4-bit NF4 Quantization Config ─────────────────────────────────────
#
# 🧠 Why 4-bit NF4?
#   • A 0.5B model in FP32 takes ~2 GB VRAM
#   • In 4-bit NF4 it takes ~0.3 GB – fits easily on a T4!
#   • NF4 (Normal Float 4) uses a distribution-aware quantization grid
#     that minimises error for normally-distributed weight values
#
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,   # quantize the quantization constants too
)
print('✅ BitsAndBytes config ready')

In [ ]:
# ── Step 8: Load Base Model in 4-bit ─────────────────────────────────────────
print(f'Loading {MODEL_ID} in 4-bit NF4...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

# How much VRAM does the model use?
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f'VRAM used after model load: {allocated:.2f} GB')

In [ ]:
# ── Step 9: Apply LoRA ────────────────────────────────────────────────────────
#
# 🧠 How LoRA works:
#   For a weight matrix W (d×k), instead of updating all d×k params,
#   we insert two small matrices: A (d×r) and B (r×k) where r << d.
#   During training: ΔW = (alpha/r) × B × A
#   At inference: W_effective = W_pretrained + ΔW
#
model = prepare_model_for_kbit_training(model)   # cast norms to float32

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # should show ~1% of params

In [ ]:
# ── Step 10: Training Arguments ───────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=ADAPTER_OUTPUT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy='epoch',
    eval_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=True,
    report_to='none',
    seed=42,
)
print('✅ Training args configured')

In [ ]:
# ── Step 11: Train! ───────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
)

print('🚀 Starting QLoRA training...')
print('This may take 10–30 minutes on a T4 GPU.')
trainer.train()

In [ ]:
# ── Step 12: Save adapter ─────────────────────────────────────────────────────
print(f'Saving adapter to: {ADAPTER_OUTPUT}')
trainer.save_model(ADAPTER_OUTPUT)
tokenizer.save_pretrained(ADAPTER_OUTPUT)
print('✅ Adapter saved!')

In [ ]:
# ── Step 13: Merge adapter and save full model ─────────────────────────────────
#
# This reloads the base model in FP16 (not quantized) and merges LoRA weights.
# The result is a standard HuggingFace model that can be converted to GGUF.
#
from peft import PeftModel

print('Loading base model for merging (FP16, no quantization)...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

print('Loading and merging LoRA adapter...')
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_OUTPUT)
merged_model = merged_model.merge_and_unload()   # folds A×B into W

MERGED_OUTPUT = 'models/merged_fp16'
Path(MERGED_OUTPUT).mkdir(parents=True, exist_ok=True)
merged_model.save_pretrained(MERGED_OUTPUT, safe_serialization=True)
tokenizer.save_pretrained(MERGED_OUTPUT)

print(f'✅ Merged model saved to: {MERGED_OUTPUT}')
print('Next step: download merged_fp16/ and run convert_gguf.sh')

In [ ]:
# ── Step 14: Quick sanity test ────────────────────────────────────────────────
print('Testing fine-tuned model...')
prompt = '### Instruction:\nWhat is the purpose of a satellite attitude control system?\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(merged_model.device)

with torch.no_grad():
    output = merged_model.generate(
        **inputs, max_new_tokens=150, temperature=0.7, do_sample=True,
        pad_token_id=tokenizer.eos_token_id)

new_tokens = output[0][inputs['input_ids'].shape[1]:]
response   = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(f'\n🤖 Response:\n{response}')

## ✅ Fine-Tuning Complete!

What you have produced:
- `models/adapter/` – the LoRA adapter weights
- `models/merged_fp16/` – the full merged FP16 model (ready for GGUF conversion)

**Next steps:**
1. Download `models/merged_fp16/` to your local machine
2. Run `src/quantization/convert_gguf.sh` to create the Q4_K_M GGUF file
3. Run `src/inference/generate.py` for local CPU inference
4. Open `04_model_evaluation.ipynb` to benchmark the models
